# 22AIE437 Medical Image Processing - Project Review 2 Upgrade
## Automated Retinal Blood Vessel Segmentation using CAR-UNet on the FIVES Benchmark

**Authors / Project Group**: Medical Image Processing Team  
**Upgrade Research Paper**: Guo et al. (IEEE 2021 / arXiv:2004.03702) *"Channel Attention Residual U-Net for Retinal Vessel Segmentation"*  
**Dataset Reference**: Jin et al. (Nature Scientific Data, 2022) *"FIVES: A Fundus Image Dataset for AI-based Vessel Segmentation"*  
**Baseline Reference**: Ronneberger et al. (2015) *"U-Net: Convolutional Networks for Biomedical Image Segmentation"*  

---

### Key Review 2 Innovations & Methodology
1. **Dataset Scaling (DRIVE -> FIVES)**: Upgraded from 40 DRIVE images to 800 high-resolution FIVES images (20x scale) across Normal, AMD, Diabetic Retinopathy, and Glaucoma cohorts.
2. **Modified Efficient Channel Attention (MECA)**: Adaptive 1D convolution across channels without dimensionality reduction.
3. **Channel Attention Double Residual Blocks (CADRB)**: Deep identity residual learning embedded with channel attention.
4. **MECA Bridge Attention**: Recalibrates encoder skip connection features before decoder concatenation.
5. **Ronneberger Overlap-Tile Strategy**: Valid convolutions (padding=0) with 284x284 to 100x100 patch reconstruction.

## 1. Setup & Environment Verification

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import sys
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import torch

sys.path.append(os.path.abspath('.'))
from src.preprocessing_fives import preprocess_image, extract_green_channel, apply_clahe, apply_noise_reduction, generate_fov_mask
from src.unet_model import UNet
from src.car_unet import CARUNet
from src.losses import CombinedBCEDiceLoss
from src.metrics import compute_fov_metrics
from src.utils import predict_full_image, save_prediction_figure

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using compute device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 2. Classical Preprocessing Pipeline Demonstration

In [ ]:
img_paths = sorted(glob.glob('data/FIVES_resized/train/images/*.png'))
mask_paths = sorted(glob.glob('data/FIVES_resized/train/masks/*.png'))

sample_idx = 0
raw_bgr = cv2.imread(img_paths[sample_idx])
raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
green = extract_green_channel(raw_rgb)
clahe = apply_clahe(green, clip_limit=2.0, tile_grid_size=(8, 8))
denoised = apply_noise_reduction(clahe, method='bilateral')
prep = preprocess_image(raw_rgb)
fov = generate_fov_mask(raw_rgb)
gt_mask = np.array(Image.open(mask_paths[sample_idx]))

fig, axes = plt.subplots(1, 6, figsize=(22, 4))
axes[0].imshow(raw_rgb); axes[0].set_title('1. Raw RGB'); axes[0].axis('off')
axes[1].imshow(green, cmap='gray'); axes[1].set_title('2. Green Channel'); axes[1].axis('off')
axes[2].imshow(clahe, cmap='gray'); axes[2].set_title('3. CLAHE Enhanced'); axes[2].axis('off')
axes[3].imshow(denoised, cmap='gray'); axes[3].set_title('4. Bilateral Filter'); axes[3].axis('off')
axes[4].imshow(prep, cmap='gray'); axes[4].set_title('5. Normalized [0, 1]'); axes[4].axis('off')
axes[5].imshow(gt_mask, cmap='gray'); axes[5].set_title('6. Ground Truth'); axes[5].axis('off')
plt.tight_layout()
plt.show()


## 3. CAR-UNet Model Architecture Verification

In [ ]:
model_car = CARUNet(in_channels=1, out_channels=1, base_filters=64).to(device)
total_params = sum(p.numel() for p in model_car.parameters())
trainable_params = sum(p.numel() for p in model_car.parameters() if p.requires_grad)

print('CAR-UNet Architecture:')
print(f'  Total Parameters:     {total_params:,}')
print(f'  Trainable Parameters: {trainable_params:,}')

dummy_in = torch.randn(2, 1, 284, 284, device=device)
dummy_out = model_car(dummy_in)
print('Input shape: ', dummy_in.shape)
print('Output shape:', dummy_out.shape, '(Expected: [2, 1, 100, 100])')


## 4. Load Trained Model Checkpoints & Run Test Inference

In [ ]:
# 1. Load U-Net on FIVES
unet_model = UNet(in_channels=1, out_channels=1, base_filters=64).to(device)
unet_ckpt = torch.load('outputs/saved_models/unet_fives.pt', map_location=device)
unet_model.load_state_dict(unet_ckpt['model_state_dict'])
unet_model.eval()
print('Loaded Scaled U-Net (FIVES) Checkpoint')

# 2. Load CAR-UNet on FIVES
car_model = CARUNet(in_channels=1, out_channels=1, base_filters=64).to(device)
car_ckpt = torch.load('outputs/saved_models/car_unet_fives.pt', map_location=device)
car_model.load_state_dict(car_ckpt['model_state_dict'])
car_model.eval()
print('Loaded CAR-UNet (FIVES) Checkpoint')


## 5. Consolidated 3-Way Benchmark Comparison Table

In [ ]:
df_comp = pd.read_csv('results/comparison_table.csv')
print(df_comp.to_string(index=False))


## 6. Visual 5-Column Side-by-Side Demonstration

In [ ]:
fig_img = Image.open('outputs/final_comparison_figure.png')
plt.figure(figsize=(24, 14))
plt.imshow(fig_img)
plt.axis('off')
plt.title('3-Way Visual Benchmark Comparison (Raw Fundus -> GT -> U-Net DRIVE -> U-Net FIVES -> CAR-UNet FIVES)', fontsize=14, fontweight='bold')
plt.show()
